# 1. Przygotowanie środowiska `.venv`

Ten notebook tworzy izolowane środowisko w katalogu `302-tts-supertonic/.venv`, instaluje pakiety z `requirements.txt` i rejestruje kernel Jupyter `Supertonic Workshop (.venv)`.

> Supertonic 1.3.1 wymaga Pythona 3.9 lub nowszego. Ten projekt został sprawdzony lokalnie także z Pythonem 3.14.

Uruchamiaj komórki po kolei. Ten notebook nie uruchamia serwera ani modelu.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

def find_project_dir() -> Path:
    current = Path.cwd().resolve()
    candidates = [current, current / "302-tts-supertonic", *current.parents]
    for candidate in candidates:
        if (candidate / "requirements.txt").is_file() and (candidate / "webgui").is_dir():
            return candidate
    raise FileNotFoundError("Nie znaleziono katalogu 302-tts-supertonic.")

PROJECT_DIR = find_project_dir()
VENV_DIR = PROJECT_DIR / ".venv"
REQUIREMENTS = PROJECT_DIR / "requirements.txt"
KERNEL_NAME = "supertonic-workshop"

print(f"Projekt: {PROJECT_DIR}")
print(f"Środowisko: {VENV_DIR}")

## Wybór zgodnej wersji Pythona

Notebook zaczyna od interpretera uruchamiającego Jupyter, a potem sprawdza typowe polecenia dla Pythona 3.13 i 3.12. Wymagana jest wersja 3.9 lub nowsza.

In [ ]:
def read_python_version(command: list[str]) -> tuple[int, int] | None:
    try:
        result = subprocess.run(
            [*command, "-c", "import sys; print(f'{sys.version_info.major}.{sys.version_info.minor}')"],
            check=True, capture_output=True, text=True,
        )
        major, minor = result.stdout.strip().split(".")
        return int(major), int(minor)
    except (FileNotFoundError, subprocess.CalledProcessError, ValueError):
        return None

candidates = [[sys.executable], ["py", "-3.13"], ["py", "-3.12"], ["python3.13"], ["python3.12"]]
PYTHON_COMMAND = next(
    (command for command in candidates if (version := read_python_version(command)) and version >= (3, 9)),
    None,
)

if PYTHON_COMMAND is None:
    raise RuntimeError("Zainstaluj Python 3.9 lub nowszy, a następnie uruchom tę komórkę ponownie.")

print("Wybrany interpreter:", " ".join(PYTHON_COMMAND), read_python_version(PYTHON_COMMAND))

## Utworzenie `.venv` i instalacja

Pierwsza instalacja pobierze biblioteki. Model Supertonic (~400 MB) zostanie pobrany dopiero przy pierwszym użyciu TTS w notebooku 2 lub na serwerze.

In [ ]:
subprocess.run([*PYTHON_COMMAND, "-m", "venv", str(VENV_DIR)], check=True)

venv_python = VENV_DIR / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
subprocess.run([str(venv_python), "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([str(venv_python), "-m", "pip", "install", "-r", str(REQUIREMENTS)], check=True)

print("Zależności zainstalowane w:", VENV_DIR)

## Rejestracja kernela Jupyter

Po wykonaniu komórki przełącz kernel notebooków 2 i 3 na **Supertonic Workshop (.venv)**.

In [ ]:
subprocess.run([
    str(venv_python), "-m", "ipykernel", "install", "--user",
    "--name", KERNEL_NAME, "--display-name", "Supertonic Workshop (.venv)",
], check=True)

subprocess.run([str(venv_python), "-m", "pip", "check"], check=True)
print("Gotowe. Teraz otwórz notebook 2 i wybierz kernel Supertonic Workshop (.venv).")